# IPC2BNS-Verify — Phase 5: Adaptivity & Refresh Simulation (Stage 4 Ablation)

This notebook demonstrates the pipeline's **zero-downtime statutory adaptivity**:
1. **Legislative Amendment Ingestion** (`injected_amendment_cases.csv`): Simulates 2025/2026 amendments (e.g. BNS §318A AI Deepfake Fraud).
2. **Incremental Index Hot-Patching** (`updater.py`): Ingests amendments and re-weights terms without re-indexing from scratch.
3. **Stage 4 Ablation Execution** (`run_stage4.py`): Compares Pre-Refresh (Stage 3) vs. Post-Refresh (Stage 4) accuracy.
4. **Automated Unit Tests**: Runs pytest suite.

---
## 1. Mount Google Drive & Environment Setup

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment initialized.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project Root: /content/drive/MyDrive/NLP_rspaper
Environment initialized.


---
## 2. Install Dependencies

In [8]:
!pip install -q pytest
print('Pytest ready.')

Pytest ready.


---
## 3. Inspect Simulated Legislative Amendments

In [9]:
import csv
amd_file = os.path.join(PROJECT_ROOT, 'data/04_refresh_sim/injected_amendment_cases.csv')

with open(amd_file, 'r') as f:
    reader = csv.DictReader(f)
    for r in reader:
        print('='*75)
        print(f'Amendment ID: {r["amendment_id"]} [{r["change_type"]}]')
        print(f'Provision   : {r["act"]} §{r["section_number"]} - {r["section_title"]}')
        print(f'Text        : {r["section_text"][:120]}...')

Amendment ID: AMD_2025_001 [NEW_SECTION]
Provision   : BNS §318A - Cheating by synthetic deepfake or generative AI impersonation
Text        : Whoever, by employing generative artificial intelligence, synthetic media, voice cloning, or deepfake technology, fraudu...
Amendment ID: AMD_2025_002 [NEW_SECTION]
Provision   : BNS §278A - Aggravated industrial pollution endangering public water supply
Text        : Whoever knowingly or negligently discharges hazardous industrial effluent or toxic pollutants into any public reservoir ...
Amendment ID: AMD_2025_003 [MODIFIED_PUNISHMENT]
Provision   : BNS §106 - Causing death by negligence (Amended)
Text        : (1) Whoever causes death by rash or negligent act not amounting to culpable homicide shall be punished with imprisonment...


---
## 4. Hot-Patch Vector Index (Incremental Refresh)

In [10]:
from src.refresh.updater import create_post_refresh_index

base_idx = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage2_index')
post_idx = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage4_post_refresh_index')

refreshed_index = create_post_refresh_index(base_idx, amd_file, post_idx)
print(f'Post-refresh snapshot saved with {refreshed_index.total_docs} statutory chunks.')

Post-refresh snapshot saved with 277 statutory chunks.


---
## 5. Interactive Pre-Refresh vs. Post-Refresh Search

In [11]:
from src.retrieval.search import StatutoryRetriever

pre_retriever = StatutoryRetriever(base_idx)
post_retriever = StatutoryRetriever(post_idx)

query = 'What is the section for AI deepfake impersonation fraud in amended BNS?'
print(f'Query: "{query}"\n')

print('--- [PRE-REFRESH INDEX RETRIEVAL] ---')
pre_hits = pre_retriever.retrieve(query, top_k=2)
for h in pre_hits:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

print('\n--- [POST-REFRESH INDEX RETRIEVAL] ---')
post_hits = post_retriever.retrieve(query, top_k=2)
for h in post_hits:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]} (Target BNS §318A Found!)')

Query: "What is the section for AI deepfake impersonation fraud in amended BNS?"

--- [PRE-REFRESH INDEX RETRIEVAL] ---
  BNS §2(24): Person
  IPC §11: Person

--- [POST-REFRESH INDEX RETRIEVAL] ---
  BNS §318A: Cheating by synthetic deepfake or generative AI impersonation (Target BNS §318A Found!)
  BNS §2(24): Person (Target BNS §318A Found!)


---
## 6. Execute Stage 4 (+Verifier+Refresh) Benchmark

In [12]:
from src.generation.run_stage4 import run_stage4_ablation

stage4_out = os.path.join(PROJECT_ROOT, 'results/stage4/stage4_refresh_results.json')
s4_data = run_stage4_ablation(base_idx, post_idx, stage4_out)

print('\n' + '='*60)
print('STAGE 4 REFRESH ADAPTIVITY SUMMARY')
print('='*60)
print(f'Pre-Refresh Retrieval Accuracy  : {s4_data["pre_refresh_retrieval_accuracy"]*100:.1f}%')
print(f'Post-Refresh Retrieval Accuracy : {s4_data["post_refresh_retrieval_accuracy"]*100:.1f}%')
print(f'Adaptivity Accuracy Delta       : +{s4_data["accuracy_delta"]*100:.1f}%')


STAGE 4 REFRESH ADAPTIVITY SUMMARY
Pre-Refresh Retrieval Accuracy  : 33.3%
Post-Refresh Retrieval Accuracy : 100.0%
Adaptivity Accuracy Delta       : +66.7%


---
## 7. Run Full Test Suite (65 Tests)

In [13]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, typeguard-4.6.0, langsmith-0.11.1
collected 65 items                                                             

drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_table_loads_successfully PASSED [  1%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_schema_columns PASSED [  3%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[302-103] PASSED [  4%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[299-100] PASSED [  6%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[304A-106] PASSED [  7%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[30

---
## 8. Check Progress against WBS

In [14]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

# Project Progress Report
**Overall: 28/32 tasks complete (88%)**

_Generated: 2026-09-03T07:39:46_

## 0. Setup — 3/4 (75%)
- [x] Repo scaffolding + config system  `(code/src, code/configs)`
- [x] India Code raw text downloaded  `(data/00_raw/india_code)`
- [ ] Concordance source PDF(s) collected  `(data/00_raw/concordance_source_pdfs)`
- [x] Data Management Plan written  `(docs/IPC2BNS-Verify_Data_Management_Plan.md)`

## 1. Mapping Module — 5/5 (100%)
- [x] Ground-truth concordance table finalized  `(data/02_ground_truth/concordance_v1.csv)`
- [x] Concordance validation report reviewed  `(data/02_ground_truth/validation_report.csv)`
- [x] Deterministic lookup function implemented  `(code/src/mapping/lookup.py)`
- [x] Query normalizer implemented  `(code/src/mapping/normalizer.py)`
- [x] Mapping module unit tests  `(code/tests/test_concordance.py)`

## 2. Ingestion & Retrieval — 6/6 (100%)
- [x] Section-level chunker implemented  `(code/src/ingestion/chunker.py)`
- [x] Cleaned sectio